In [ ]:
!pip install pandas numpy scikit-learn nltk gensim tensorflow keras matplotlib seaborn
!pip install pymorphy3 wordcloud

# Импорт библиотек

In [ ]:
import pandas as pd
import numpy as np
import re

# Библиотеки для обработки текста

In [ ]:
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import pymorphy3

# Скачивание необходимых ресурсов NLTK

In [ ]:
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('averaged_perceptron_tagger')

# Векторизация

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from gensim.models import Word2Vec, FastText
from gensim.models.keyedvectors import KeyedVectors
import gensim.downloader as api

# Модели машинного обучения

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC

# Нейронные сети

In [ ]:
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, LSTM, Embedding, Conv1D, GlobalMaxPooling1D
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Метрики и утилиты

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# 1. ЗАГРУЗКА И ПОДГОТОВКА ДАННЫХ

Для демонстрации используем датасет с новостными статьями `fetch_20newsgroups`

In [ ]:
from sklearn.datasets import fetch_20newsgroups
newsgroups = fetch_20newsgroups(subset='all', categories=['alt.atheism', 'soc.religion.christian',
                                                           'comp.graphics', 'sci.med'])
texts = newsgroups.data[:2000]  # Берем 2000 документов для скорости
labels = newsgroups.target[:2000]

print(f"Загружено {len(texts)} текстов")
print(f"Количество классов: {len(np.unique(labels))}")

# 2. ФУНКЦИИ ПРЕДОБРАБОТКИ ТЕКСТА

In [ ]:
# Инициализация морфологического анализатора
morph = pymorphy3.MorphAnalyzer()
stop_words = set(stopwords.words('russian') + stopwords.words('english'))

def basic_preprocessing(text):
    """Базовая предобработка: lowercase, удаление символов"""
    text = text.lower()
    text = re.sub(r'[^a-zA-Zа-яА-Я\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def advanced_preprocessing(text):
    """Продвинутая предобработка: лемматизация, удаление стоп-слов"""
    text = basic_preprocessing(text)
    tokens = word_tokenize(text)

    # Удаление стоп-слов
    tokens = [word for word in tokens if word not in stop_words and len(word) > 2]

    # Лемматизация
    lemmatized = []
    for word in tokens:
        parsed = morph.parse(word)[0]
        lemmatized.append(parsed.normal_form)

    return ' '.join(lemmatized)

# Применение предобработки
print("Предобработка текстов...")
texts_basic = [basic_preprocessing(text) for text in texts]
texts_advanced = [advanced_preprocessing(text) for text in texts]

# 3. ВЕКТОРИЗАЦИЯ ТЕКСТОВ

In [ ]:
def create_tfidf_vectors(texts, max_features=5000):
    """TF-IDF векторизация"""
    vectorizer = TfidfVectorizer(max_features=max_features, ngram_range=(1, 2))
    vectors = vectorizer.fit_transform(texts)
    return vectors, vectorizer

def create_word2vec_vectors(texts, vector_size=100, window=5, min_count=2):
    """Word2Vec векторизация"""
    tokenized = [text.split() for text in texts]
    model = Word2Vec(sentences=tokenized, vector_size=vector_size,
                     window=window, min_count=min_count, workers=4)

    # Усреднение векторов слов для получения вектора документа
    vectors = []
    for tokens in tokenized:
        word_vecs = [model.wv[word] for word in tokens if word in model.wv]
        if word_vecs:
            vectors.append(np.mean(word_vecs, axis=0))
        else:
            vectors.append(np.zeros(vector_size))

    return np.array(vectors), model

def create_fasttext_vectors(texts, vector_size=100, window=5, min_count=2):
    """FastText векторизация"""
    tokenized = [text.split() for text in texts]
    model = FastText(sentences=tokenized, vector_size=vector_size,
                     window=window, min_count=min_count, workers=4)

    vectors = []
    for tokens in tokenized:
        word_vecs = [model.wv[word] for word in tokens if word in model.wv]
        if word_vecs:
            vectors.append(np.mean(word_vecs, axis=0))
        else:
            vectors.append(np.zeros(vector_size))

    return np.array(vectors), model

def load_glove_vectors(texts, model_name='glove-wiki-gigaword-100'):
    """GloVe векторизация (загрузка предобученной модели)"""
    print(f"Загрузка предобученной модели {model_name}...")
    try:
        glove_model = api.load(model_name)
    except:
        print("Ошибка загрузки GloVe. Используем Word2Vec вместо этого.")
        return create_word2vec_vectors(texts)

    vectors = []
    for text in texts:
        tokens = text.split()
        word_vecs = [glove_model[word] for word in tokens if word in glove_model]
        if word_vecs:
            vectors.append(np.mean(word_vecs, axis=0))
        else:
            vectors.append(np.zeros(100))

    return np.array(vectors), glove_model

# 4. СОЗДАНИЕ РАЗЛИЧНЫХ КОМБИНАЦИЙ ПРЕДОБРАБОТКИ И ВЕКТОРИЗАЦИИ

In [ ]:
print("\nСоздание векторных представлений...")

# Комбинация 1: TF-IDF без предобработки
X_tfidf_basic, tfidf_vectorizer_basic = create_tfidf_vectors(texts_basic)
print("✓ Комбинация 1: TF-IDF + базовая предобработка")

# Комбинация 2: TF-IDF с продвинутой предобработкой
X_tfidf_advanced, tfidf_vectorizer_advanced = create_tfidf_vectors(texts_advanced)
print("✓ Комбинация 2: TF-IDF + продвинутая предобработка")

# Комбинация 3: Word2Vec без предобработки
X_w2v_basic, w2v_model_basic = create_word2vec_vectors(texts_basic)
print("✓ Комбинация 3: Word2Vec + базовая предобработка")

# Комбинация 4: Word2Vec с продвинутой предобработкой
X_w2v_advanced, w2v_model_advanced = create_word2vec_vectors(texts_advanced)
print("✓ Комбинация 4: Word2Vec + продвинутая предобработка")

# Комбинация 5: FastText с продвинутой предобработкой
X_fasttext_advanced, fasttext_model = create_fasttext_vectors(texts_advanced)
print("✓ Комбинация 5: FastText + продвинутая предобработка")

# Комбинация 6: GloVe с базовой предобработкой (опционально)
# X_glove_basic, glove_model = load_glove_vectors(texts_basic)
# print("✓ Комбинация 6: GloVe + базовая предобработка")

# 5. ОПРЕДЕЛЕНИЕ МОДЕЛЕЙ МАШИННОГО ОБУЧЕНИЯ

In [ ]:
def get_ml_models():
    """Возвращает словарь с классическими ML моделями"""
    return {
        'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
        'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
        'SVM': SVC(kernel='rbf', random_state=42),
        'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42)
    }

def create_neural_network(input_dim, num_classes):
    """Создает простую нейронную сеть"""
    model = Sequential([
        Dense(256, activation='relu', input_dim=input_dim),
        Dropout(0.5),
        Dense(128, activation='relu'),
        Dropout(0.3),
        Dense(num_classes, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    return model

def create_lstm_network(vocab_size, num_classes, max_length=200):
    """Создает LSTM нейронную сеть"""
    model = Sequential([
        Embedding(vocab_size, 128, input_length=max_length),
        LSTM(128, dropout=0.2, recurrent_dropout=0.2),
        Dense(64, activation='relu'),
        Dropout(0.5),
        Dense(num_classes, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    return model

def create_cnn_network(vocab_size, num_classes, max_length=200):
    """Создает CNN нейронную сеть для текста"""
    model = Sequential([
        Embedding(vocab_size, 128, input_length=max_length),
        Conv1D(128, 5, activation='relu'),
        GlobalMaxPooling1D(),
        Dense(64, activation='relu'),
        Dropout(0.5),
        Dense(num_classes, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    return model

# 6. ОБУЧЕНИЕ И ОЦЕНКА МОДЕЛЕЙ

In [ ]:
def evaluate_model(model, X_train, X_test, y_train, y_test, model_name, is_neural=False):
    """Обучает и оценивает модель"""
    if is_neural:
        model.fit(X_train, y_train, epochs=10, batch_size=32,
                 validation_split=0.1, verbose=0)
        y_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(y_test, y_pred, average='weighted')

    return {
        'Model': model_name,
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1-Score': f1
    }

# 7. ЗАПУСК ЭКСПЕРИМЕНТОВ

In [ ]:
# Подготовка словаря с комбинациями
combinations = {
    'TF-IDF + Basic': X_tfidf_basic,
    'TF-IDF + Advanced': X_tfidf_advanced,
    'Word2Vec + Basic': X_w2v_basic,
    'Word2Vec + Advanced': X_w2v_advanced,
    'FastText + Advanced': X_fasttext_advanced,
    # 'GloVe + Basic': X_glove_basic,  # Раскомментируйте если используете
}

# Хранилище результатов
results = []
num_classes = len(np.unique(labels))

print("\n" + "="*80)
print("НАЧАЛО ОБУЧЕНИЯ МОДЕЛЕЙ")
print("="*80)

for comb_name, X in combinations.items():
    print(f"\n{'='*80}")
    print(f"Комбинация: {comb_name}")
    print(f"{'='*80}")

    # Разделение на train/test
    X_train, X_test, y_train, y_test = train_test_split(
        X, labels, test_size=0.2, random_state=42, stratify=labels
    )

    # Классические ML модели
    ml_models = get_ml_models()
    for model_name, model in ml_models.items():
        print(f"  Обучение {model_name}...")
        result = evaluate_model(model, X_train, X_test, y_train, y_test,
                               model_name, is_neural=False)
        result['Combination'] = comb_name
        results.append(result)
        print(f"    Accuracy: {result['Accuracy']:.4f}, F1: {result['F1-Score']:.4f}")

    # Нейронные сети (для векторных представлений)
    print(f"  Обучение Dense Neural Network...")
    if len(X_train.shape) == 1:
        X_train_dense = X_train.reshape(-1, 1)
        X_test_dense = X_test.reshape(-1, 1)
    else:
        X_train_dense = X_train.toarray() if hasattr(X_train, 'toarray') else X_train
        X_test_dense = X_test.toarray() if hasattr(X_test, 'toarray') else X_test

    nn_model = create_neural_network(X_train_dense.shape[1], num_classes)
    result = evaluate_model(nn_model, X_train_dense, X_test_dense,
                           y_train, y_test, 'Dense NN', is_neural=True)
    result['Combination'] = comb_name
    results.append(result)
    print(f"    Accuracy: {result['Accuracy']:.4f}, F1: {result['F1-Score']:.4f}")

# LSTM и CNN на исходных текстах
print(f"\n{'='*80}")
print("Обучение LSTM и CNN на последовательностях")
print(f"{'='*80}")

# Подготовка данных для LSTM/CNN
tokenizer = Tokenizer(num_words=10000)
tokenizer.fit_on_texts(texts_advanced)
sequences = tokenizer.texts_to_sequences(texts_advanced)
max_length = 200
X_seq = pad_sequences(sequences, maxlen=max_length)

X_train_seq, X_test_seq, y_train_seq, y_test_seq = train_test_split(
    X_seq, labels, test_size=0.2, random_state=42, stratify=labels
)

# LSTM
print("  Обучение LSTM...")
lstm_model = create_lstm_network(10000, num_classes, max_length)
result = evaluate_model(lstm_model, X_train_seq, X_test_seq,
                       y_train_seq, y_test_seq, 'LSTM', is_neural=True)
result['Combination'] = 'Sequences + Advanced'
results.append(result)
print(f"    Accuracy: {result['Accuracy']:.4f}, F1: {result['F1-Score']:.4f}")

# CNN
print("  Обучение CNN...")
cnn_model = create_cnn_network(10000, num_classes, max_length)
result = evaluate_model(cnn_model, X_train_seq, X_test_seq,
                       y_train_seq, y_test_seq, 'CNN', is_neural=True)
result['Combination'] = 'Sequences + Advanced'
results.append(result)
print(f"    Accuracy: {result['Accuracy']:.4f}, F1: {result['F1-Score']:.4f}")

# 8. ВИЗУАЛИЗАЦИЯ РЕЗУЛЬТАТОВ

In [ ]:
# Создание DataFrame с результатами
results_df = pd.DataFrame(results)
print("\n" + "="*80)
print("ИТОГОВЫЕ РЕЗУЛЬТАТЫ")
print("="*80)
print(results_df.to_string(index=False))

# График 1: Сравнение точности по комбинациям
plt.figure(figsize=(16, 8))
pivot_accuracy = results_df.pivot(index='Model', columns='Combination', values='Accuracy')
pivot_accuracy.plot(kind='bar', figsize=(16, 8))
plt.title('Сравнение точности (Accuracy) моделей по различным комбинациям', fontsize=14, pad=20)
plt.xlabel('Модель', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.legend(title='Комбинация', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.grid(axis='y', alpha=0.3)
plt.show()

# График 2: Сравнение F1-Score
plt.figure(figsize=(16, 8))
pivot_f1 = results_df.pivot(index='Model', columns='Combination', values='F1-Score')
pivot_f1.plot(kind='bar', figsize=(16, 8))
plt.title('Сравнение F1-Score моделей по различным комбинациям', fontsize=14, pad=20)
plt.xlabel('Модель', fontsize=12)
plt.ylabel('F1-Score', fontsize=12)
plt.legend(title='Комбинация', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.grid(axis='y', alpha=0.3)
plt.show()

# График 3: Тепловая карта результатов
plt.figure(figsize=(14, 10))
heatmap_data = results_df.pivot(index='Model', columns='Combination', values='Accuracy')
sns.heatmap(heatmap_data, annot=True, fmt='.3f', cmap='YlGnBu', cbar_kws={'label': 'Accuracy'})
plt.title('Тепловая карта точности моделей', fontsize=14, pad=20)
plt.xlabel('Комбинация предобработки и векторизации', fontsize=12)
plt.ylabel('Модель', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

# График 4: Лучшие модели по каждой метрике
plt.figure(figsize=(14, 6))
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
best_models = {}

for metric in metrics:
    best_idx = results_df[metric].idxmax()
    best_models[metric] = results_df.loc[best_idx]

best_df = pd.DataFrame(best_models).T
best_df[metrics].plot(kind='bar', figsize=(14, 6))
plt.title('Лучшие результаты по каждой метрике', fontsize=14, pad=20)
plt.xlabel('Метрика', fontsize=12)
plt.ylabel('Значение', fontsize=12)
plt.legend(['Accuracy', 'Precision', 'Recall', 'F1-Score'])
plt.xticks(rotation=0)
plt.ylim([0, 1])
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()